# Taller 2 — Reformulación del modelo de programación de lotes de tomate
Este cuaderno reproduce de principio a fin la tabla del informe: genera las instancias E1–E4 con semilla fija, resuelve las formulaciones A y B con HiGHS y reporta los siete indicadores.

**Tiempo estimado de ejecución completa:** puede superar una hora (límite de 600 s × 3 repeticiones × 8 combinaciones en el peor caso). Para una prueba rápida, cambiar `LIMITE_TIEMPO` y `REPETICIONES` en la celda siguiente.

In [1]:
import os
os.environ["LIMITE_TIEMPO"] = "600"   # segundos por corrida
os.environ["REPETICIONES"] = "3"

import pandas as pd
from datos import generar, TAMANOS, SEMILLA
import experimento as ex
print("Semilla:", SEMILLA)
ex.entorno()

Semilla: 2026


{'python': '3.12.14',
 'sistema': 'macOS-26.6.2-arm64-arm-64bit',
 'procesador': 'arm',
 'pyomo': '6.10.1',
 'highspy': '1.15.1',
 'hilos': 1,
 'limite_tiempo_s': 600.0,
 'repeticiones': 3}

## 1. Instancias

In [3]:
filas = []
for nombre, n in TAMANOS.items():
    d = generar(n)
    brutas = sum(d["G"].values())
    comerciales = sum(d["Q"].values())
    filas.append({"escenario": nombre, "lotes": n, "gamelas_brutas": brutas,
                  "gamelas_comerciales_esperadas": round(comerciales),
                  "oferta_capacidad_pct": round(100 * brutas / 9600, 1)})
pd.DataFrame(filas)

,escenario,lotes,gamelas_brutas,gamelas_comerciales_esperadas,oferta_capacidad_pct
0,E1,40,4695,4202,48.9
1,E2,80,8899,8049,92.7
2,E3,160,18356,16467,191.2
3,E4,320,37259,33618,388.1


## 2. Experimento

In [4]:
resultados = []
for nombre, n in TAMANOS.items():
    print("Resolviendo", nombre)
    resultados += ex.correr_instancia(nombre, n)
df = pd.DataFrame(resultados)
df.to_csv("resultados.csv", index=False)

Resolviendo E1
Resolviendo E2
Resolviendo E3
Resolviendo E4


## 3. Verificación numérica de equivalencia
Los óptimos enteros de A y B deben coincidir en cada instancia resuelta a optimalidad. Si una formulación se detuvo por límite de tiempo, se verifica que sus intervalos [incumbente, cota] se intersecten.

In [5]:
for nombre in TAMANOS:
    a = df[(df.instancia == nombre) & (df.formulacion == "A")].iloc[0]
    b = df[(df.instancia == nombre) & (df.formulacion == "B")].iloc[0]
    if a.estado == "Optimal" and b.estado == "Optimal":
        ok = abs(a.Z_IP - b.Z_IP) <= 1e-4 * abs(a.Z_IP)
        print(nombre, "óptimos iguales:", ok, a.Z_IP, b.Z_IP)
        assert ok, "Los óptimos enteros difieren: la comparación no es válida"
    else:
        ok = max(a.Z_IP, b.Z_IP) <= min(a.cota_final, b.cota_final) + 1e-6
        print(nombre, "intervalos compatibles (no certificado por límite de tiempo):", ok)

E1 óptimos iguales: True 2952187.813782705 2952187.8137827143
E2 óptimos iguales: True 5439896.906492326 5439896.906492333
E3 óptimos iguales: True 6409992.37906997 6409992.379069969
E4 óptimos iguales: True 6714249.624672406 6714249.624672441


## 4. Tabla del informe

In [6]:
tabla = df[["instancia", "oferta_pct", "formulacion", "variables", "restricciones", "no_nulos",
            "Z_LP", "Z_IP", "brecha_LP_pct", "integrality_gap", "brecha_opt_pct",
            "nodos", "cortes_generados", "cortes_activos", "tiempo_medio_s", "estado"]].copy()
tabla.round({"Z_LP": 0, "Z_IP": 0, "brecha_LP_pct": 2, "integrality_gap": 4,
             "brecha_opt_pct": 4, "tiempo_medio_s": 2})

,instancia,oferta_pct,formulacion,variables,restricciones,no_nulos,Z_LP,Z_IP,brecha_LP_pct,integrality_gap,brecha_opt_pct,nodos,cortes_generados,cortes_activos,tiempo_medio_s,estado
0,E1,48.9,A,1004,408,2924,3027555.0,2952188.0,2.55,1.0255,0.0000,25,856,31,0.78,Optimal
1,E1,48.9,B,1004,608,3924,3027555.0,2952188.0,2.55,1.0255,0.0000,17,665,41,0.69,Optimal
2,E2,92.7,A,2004,808,5844,5643791.0,5439897.0,3.75,1.0375,0.0000,621,3773,312,4.58,Optimal
3,E2,92.7,B,2004,1208,7844,5643791.0,5439897.0,3.75,1.0375,0.0000,9,994,11,1.32,Optimal
4,E3,191.2,A,4004,1608,11684,8045974.0,6409992.0,25.52,1.2552,0.0001,682,7008,1204,19.10,Optimal
5,E3,191.2,B,4004,2408,15684,6520022.0,6409992.0,1.72,1.0172,0.0000,380,1085,53,5.74,Optimal
6,E4,388.1,A,8004,3208,23364,8323533.0,6714250.0,23.97,1.2397,0.0001,6354,11910,2733,153.94,Optimal
7,E4,388.1,B,8004,4808,31364,6807089.0,6714250.0,1.38,1.0138,0.0001,1689,2424,1685,28.46,Optimal
